In [ ]:
#!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 17.4 MB/s  0:00:00


In [1]:
import pandas as pd
import os
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from tqdm import tqdm

In [ ]:
# 1. Initialization & Setup
# Activate pandas progress bar
tqdm.pandas()

try:
    nltk.data.find('vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon')

print("Initializing VADER and loading custom Airbnb lexicon...")
SIA = SentimentIntensityAnalyzer()

# 
custom_lexicon = {
    'cancel': -20, 'canceled': -20, 'canceling': -20,
    'backpain': -5, 'bad': -10, 'spiderwebs': -50,
    'odor': -30, 'freaked': -30, 'musty': -50,
    'toxic': -5, 'sticky': -15, 'ugly': -15,
    'bedbugs': -60, 'bugs': -20, 'rude': -30,
    'aggressive': -30, 'scary': -15, 'cozy': 10,
    'great': 20, 'cosy': 10, 'smoothly': 30,
    'worst': -10, 'convenient': 40, 'worse': -10,
    'exciting': 60, 'notch': 30, 'superhost': 30,
    'disappointing': -10, 'horrible': -30, 'dirty': -15,
    'dirt': -15, 'stain': -20, 'filthy': -30,
    'unreliable': -15, 'meh': -5, 'spacious': 10,
    'lovely': 20, 'infested': -15, 'broke': -10,
    'broken': -15, 'awake': -20, 'difficult': -20,
    '1010': 10
}
SIA.lexicon.update(custom_lexicon)

Initializing VADER and loading custom Airbnb lexicon...


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/gonghuan/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
# 2. Optimized Functions

def clean_text_optimized(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = ' '.join(text.split())
    return text

def get_vader_score(sentence):
    return SIA.polarity_scores(sentence)['compound']

In [ ]:
# 3. Main Execution Pipeline

if __name__ == "__main__":
    print("\n--- Starting Large-Scale Analysis Pipeline ---")

    folder_path = "/Users/gonghuan/Downloads/MergedEdinburgh"
    target_filename = "2024-08-17_reviews.csv.gz" 
    full_file_path = os.path.join(folder_path, target_filename)
    
    if not os.path.exists(full_file_path):
        print(f"[Error] File not found: {full_file_path}")
    else:
        # 2. Load Data
        df = pd.read_csv(full_file_path, usecols=['listing_id', 'id', 'comments'])
        print(f"✅ Successfully loaded initial data: {len(df)} rows.\n")

        # Step 1: Text Cleaning & Filtering
       
        print("--- Step 1/3: Cleaning text & Filtering ---")
        
        # 1.1 Drop NaNs
        initial_len = len(df)
        df = df.dropna(subset=['comments'])
        dropped_nans = initial_len - len(df)
        print(f"🗑️ [Filter] Dropped {dropped_nans} rows due to missing comments. Rows remaining: {len(df)}")

        # 1.2 Text Cleaning
        print("⏳ Applying text cleaning (HTML strip, space normalization)...")
        df['clean_text'] = df['comments'].progress_apply(clean_text_optimized)
        
        print("\n🔍 [Sample] Text Cleaning Changes (Before vs After):")
        changed_samples = df[df['comments'] != df['clean_text']].head(3)
        if not changed_samples.empty:
            for idx, row in changed_samples.iterrows():
                print(f"  [-] Before : {repr(row['comments'][:100])}...")
                print(f"  [+] After  : {repr(row['clean_text'][:100])}...\n")
        else:
            print("  No visible changes in the first few rows.\n")
        
        # 1.3 Drop short comments
        pre_short_len = len(df)
        df = df[df['clean_text'].str.len() > 3]
        dropped_short = pre_short_len - len(df)
        print(f"🗑️ [Filter] Dropped {dropped_short} rows due to short length (<= 3 chars). Rows remaining: {len(df)}\n")

        # Step 2: Sentiment Analysis
        print("--- Step 2/3: Running Custom VADER Sentiment Analysis ---")
        df['sentiment_score'] = df['clean_text'].progress_apply(get_vader_score)

        print("\n🔍 [Sample] Sentiment Scoring Results:")
        for idx, row in df.head(3).iterrows():
            print(f"  [Text]  : {repr(row['clean_text'][:100])}...")
            print(f"  [Score] : {row['sentiment_score']}\n")

        # Step 3: Aggregation
        print("--- Step 3/3: Aggregating results per listing ---")
        initial_listings = df['listing_id'].nunique()
        print(f"📊 Total unique listings before aggregation: {initial_listings}")

        listing_sentiment = df.groupby('listing_id').agg({
            'sentiment_score': 'mean',
            'id': 'count'
        }).rename(columns={'sentiment_score': 'average_sentiment', 'id': 'review_count'})

        # Keep listings with statistically significant review counts (e.g., >= 5)
        pre_filter_listings = len(listing_sentiment)
        listing_sentiment = listing_sentiment[listing_sentiment['review_count'] >= 5]
        listing_sentiment = listing_sentiment.sort_values(by='average_sentiment', ascending=False)
        
        dropped_listings = pre_filter_listings - len(listing_sentiment)
        print(f"🗑️ [Filter] Dropped {dropped_listings} listings with < 5 reviews.")

        


--- Starting Large-Scale Analysis Pipeline ---
✅ Successfully loaded initial data: 551523 rows.

--- Step 1/3: Cleaning text & Filtering ---
🗑️ [Filter] Dropped 44 rows due to missing comments. Rows remaining: 551479
⏳ Applying text cleaning (HTML strip, space normalization)...


100%|██████████| 551479/551479 [00:02<00:00, 240568.94it/s]



🔍 [Sample] Text Cleaning Changes (Before vs After):
  [-] Before : 'My wife and I stayed at this beautiful apartment and our stay was spectacular.  The neighborhood is '...
  [+] After  : 'My wife and I stayed at this beautiful apartment and our stay was spectacular. The neighborhood is v'...

  [-] Before : "Charlotte couldn't have been a more thoughtful or accomodating host.\r<br/>The flat had literally eve"...
  [+] After  : "Charlotte couldn't have been a more thoughtful or accomodating host. The flat had literally everythi"...

  [-] Before : "This flat was incredible. As other guests have mentioned, Charlotte's attention to detail is unrival"...
  [+] After  : "This flat was incredible. As other guests have mentioned, Charlotte's attention to detail is unrival"...

🗑️ [Filter] Dropped 1659 rows due to short length (<= 3 chars). Rows remaining: 549820

--- Step 2/3: Running Custom VADER Sentiment Analysis ---


100%|██████████| 549820/549820 [02:08<00:00, 4266.38it/s]


🔍 [Sample] Sentiment Scoring Results:
  [Text]  : 'My wife and I stayed at this beautiful apartment and our stay was spectacular. The neighborhood is v'...
  [Score] : 0.9816

  [Text]  : "Charlotte couldn't have been a more thoughtful or accomodating host. The flat had literally everythi"...
  [Score] : 0.8583

  [Text]  : "I went to Edinburgh for the second time on April 2011 and, also thanks to Charlotte's flat, the stay"...
  [Score] : 0.9959

--- Step 3/3: Aggregating results per listing ---
📊 Total unique listings before aggregation: 5215
🗑️ [Filter] Dropped 526 listings with < 5 reviews.


In [ ]:
# 4. Save Results
print(f"\n✅ Analysis Complete. {len(listing_sentiment)} valid listings remaining.")

output_dir = "/Users/gonghuan/Downloads"
output_filename = "airbnb_sentiment_results_ml_202408.csv"

output_path = os.path.join(output_dir, output_filename)

listing_sentiment.to_csv(output_path)
print(f"💾 Results successfully saved to: {output_path}")


✅ Analysis Complete. 4689 valid listings remaining.
💾 Results successfully saved to: /Users/gonghuan/Downloads/airbnb_sentiment_results_ml.csv
